# Experiment 3.16: Eigenvalue Lifting -- Per-Step vs Cumulative

## Motivation and Scientific Context

Experiment 2.10 demonstrated a dramatic **330x reduction** in the Hessian condition number (kappa = lambda_max / lambda_min) when comparing Muon to SGD at convergence. This is a striking result, but it leaves a fundamental causal question unanswered:

> **Is the kappa reduction a per-step effect of Muon's orthogonalized updates, or is it purely a cumulative consequence of many Muon steps compounding over training?**

This distinction matters because it speaks to the **mechanism** of Muon's spectral regularization:
- If **per-step**: each Muon update inherently reshapes the loss landscape by lifting the smallest Hessian eigenvalue (lambda_min) more than SGD does. The orthogonal projection via Newton-Schulz iteration acts as a single-step spectral equalizer.
- If **cumulative**: individual Muon steps are not dramatically different from SGD steps, but the trajectory they carve through parameter space gradually finds better-conditioned regions -- an emergent, path-dependent effect.

## Experimental Design

We use a **2-layer 4x4 deep linear network** (32 parameters total), small enough to compute the full 32x32 Hessian exactly via finite differences. The protocol is:

1. **Warmup phase**: Train with SGD for up to 1500 steps, saving weight snapshots at 10 measurement points.
2. **Causal comparison**: From each snapshot (same starting weights, same momentum state):
   - Take **one SGD step** --> compute full Hessian H_after_SGD
   - Take **one Muon step** --> compute full Hessian H_after_Muon
   - Compare both to H_before (Hessian at the snapshot)
3. **Repeat** over 5 random seeds for statistical robustness.

## Key Metrics
- **lambda_min lifting ratio**: `lambda_min(H_after) / lambda_min(H_before)` -- values > 1 mean the optimizer lifted the floor of the spectrum
- **kappa change ratio**: `kappa(H_after) / kappa(H_before)` -- values < 1 mean conditioning improved
- **trace ratio**: `tr(H_after) / tr(H_before)` -- tracks total curvature change
- **Muon/SGD lifting ratio**: the critical test statistic; values > 3 confirm a strong per-step effect

## Key Hypothesis Test
**T2 (critical)**: Does Muon lift lambda_min by more than **3x** what SGD does, per step, on average across all measurement points?

In [ ]:
import numpy as np
import os

print("NumPy version:", np.__version__)
print("Imports loaded successfully.")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

## Configuration

### Why these specific parameters?

- **DIM=4, NUM_LAYERS=2**: This gives a 32-parameter network where the full 32x32 Hessian can be computed exactly via finite differences in ~2*32 = 64 gradient evaluations per Hessian. Larger networks would make full Hessian computation prohibitive.
- **HESSIAN_EPS=1e-5**: Central finite difference step size. Too large introduces truncation error; too small introduces floating-point cancellation. 1e-5 is a standard sweet spot for double precision.
- **LR_MUON=0.02 vs LR_SGD=0.01**: Muon uses 2x the learning rate because orthogonalized gradients have unit spectral norm by construction -- the effective step scale is already normalized, so a higher LR is needed for a fair comparison of optimizer dynamics (not just step magnitude).
- **NS_ITERS=5**: Five iterations of the quintic Newton-Schulz iteration converge to machine precision for the 4x4 matrices in this experiment.
- **MEASUREMENT_STEPS**: 10 points spanning early (50) to late (1300) training, allowing us to track whether the per-step effect changes as the network approaches convergence.
- **NUM_SEEDS=5**: Enough for reasonable variance estimation while keeping compute manageable.

In [ ]:
DIM = 4
NUM_LAYERS = 2
N_PARAMS = NUM_LAYERS * DIM * DIM  # 32
HESSIAN_EPS = 1e-5
DATA_POINTS = 32
MOMENTUM = 0.9
LR_SGD = 0.01
LR_MUON = 0.02
NS_ITERS = 5
WARMUP_MAX_STEPS = 1500
NUM_SEEDS = 5

print(f"Network architecture: {NUM_LAYERS}-layer deep linear net, {DIM}x{DIM} weight matrices")
print(f"Total parameters: {N_PARAMS}")
print(f"Hessian size: {N_PARAMS}x{N_PARAMS} = {N_PARAMS**2} entries")
print(f"Hessian finite-diff epsilon: {HESSIAN_EPS}")
print(f"Training data: {DATA_POINTS} samples of dimension {DIM}")
print(f"SGD learning rate: {LR_SGD}, Muon learning rate: {LR_MUON} (ratio: {LR_MUON/LR_SGD:.1f}x)")
print(f"Momentum coefficient: {MOMENTUM}")
print(f"Newton-Schulz iterations for Muon: {NS_ITERS}")
print(f"Warmup training length: {WARMUP_MAX_STEPS} SGD steps")
print(f"Number of random seeds: {NUM_SEEDS}")

In [ ]:
# Measurement points: step indices during warmup at which to snapshot
MEASUREMENT_STEPS = [50, 100, 150, 200, 300, 400, 600, 800, 1000, 1300]

print(f"Measurement points ({len(MEASUREMENT_STEPS)} snapshots): {MEASUREMENT_STEPS}")
print(f"  Early training (steps < 200):  {[s for s in MEASUREMENT_STEPS if s < 200]}")
print(f"  Mid training (200-600):        {[s for s in MEASUREMENT_STEPS if 200 <= s <= 600]}")
print(f"  Late training (steps > 600):   {[s for s in MEASUREMENT_STEPS if s > 600]}")
print(f"\nTotal Hessian computations per seed: {len(MEASUREMENT_STEPS)} snapshots x 3 Hessians (before/SGD/Muon) = {len(MEASUREMENT_STEPS) * 3}")
print(f"Total Hessians across all seeds: {len(MEASUREMENT_STEPS) * 3 * NUM_SEEDS}")

## Network Utilities

### Deep Linear Network

We use a deep linear network: `y = W_2 @ W_1 @ x`. Despite having no nonlinearities, deep linear networks exhibit rich optimization dynamics -- the loss landscape has saddle points, degenerate curvature, and nontrivial Hessian structure that depends on the **product** `W_2 @ W_1` rather than individual factors. This makes them an ideal testbed for studying how optimizers interact with curvature.

**Initialization near identity** (`W = I + 0.1 * noise`) ensures the initial product `W_2 W_1` is close to identity, placing us in a well-conditioned starting region from which training can degrade or improve conditioning.

In [ ]:
def init_weights(dim, num_layers, seed):
    """Initialize layers near identity."""
    rng = np.random.RandomState(seed)
    weights = []
    for _ in range(num_layers):
        W = np.eye(dim) + rng.randn(dim, dim) * 0.1
        weights.append(W.copy())
    return weights

In [ ]:
def forward_linear(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y_target):
    Y_pred = forward_linear(weights, X)
    diff = Y_pred - Y_target
    return 0.5 * np.mean(diff ** 2)

In [ ]:
def compute_gradients(weights, X, Y_target):
    num_layers = len(weights)
    batch_size = X.shape[1]

    activations = [X.copy()]
    for W in weights:
        activations.append(W @ activations[-1])

    delta = (activations[-1] - Y_target) / batch_size

    grads = [None] * num_layers
    for l in range(num_layers - 1, -1, -1):
        grads[l] = delta @ activations[l].T
        if l > 0:
            delta = weights[l].T @ delta

    return grads

## Newton-Schulz Orthogonalization

### The Core of Muon's Mechanism

The Newton-Schulz (NS) iteration is the heart of the Muon optimizer. Given a gradient matrix G, it computes the closest orthogonal matrix (in the polar decomposition sense) via a purely matrix-multiplication-based iteration -- no SVD needed.

**Quintic variant** (`X_{k+1} = a*X + b*X(X^T X) + c*X(X^T X)^2`) with coefficients `a=3.4445, b=-4.7750, c=2.0315` converges to the orthogonal factor in ~5 iterations for well-conditioned inputs.

**Why orthogonalization matters for eigenvalue lifting**: By projecting the gradient onto the Stiefel manifold (set of orthogonal matrices), Muon ensures that the update direction has **equal magnitude across all singular directions**. This means the update "pushes" equally along flat and steep directions of the loss landscape, effectively applying more relative force to the directions with small Hessian eigenvalues -- precisely the mechanism that could lift lambda_min per step.

In [ ]:
def newton_schulz_orthogonalize(G, num_iters=5):
    """Quintic Newton-Schulz: a=3.4445, b=-4.7750, c=2.0315."""
    a, b, c = 3.4445, -4.7750, 2.0315
    norm = np.linalg.norm(G, 'fro')
    if norm < 1e-12:
        return G
    X = G / norm

    for _ in range(num_iters):
        XtX = X.T @ X
        X_XtX = X @ XtX
        XtX2 = XtX @ XtX
        X_XtX2 = X @ XtX2
        X = a * X + b * X_XtX + c * X_XtX2

    return X

## Full Hessian Computation

### Exact Curvature via Central Finite Differences

For this small network (32 parameters), we can compute the **complete** 32x32 Hessian matrix H, where `H[i,j] = d^2 L / (d theta_i d theta_j)`. We use central finite differences on the gradient:

```
H[:, i] = (grad(theta + eps*e_i) - grad(theta - eps*e_i)) / (2*eps)
```

This gives O(eps^2) accuracy. The result is symmetrized via `H = 0.5*(H + H^T)` to eliminate any numerical asymmetry.

**Why full Hessian matters**: Unlike stochastic Hessian-vector products or trace estimators, the full Hessian gives us the **complete eigenspectrum** -- every eigenvalue, including the critical lambda_min that determines conditioning. This is the gold standard for curvature analysis.

In [ ]:
def weights_to_vector(weights):
    return np.concatenate([W.flatten() for W in weights])

In [ ]:
def vector_to_weights(vec, shapes):
    weights = []
    idx = 0
    for shape in shapes:
        size = shape[0] * shape[1]
        W = vec[idx:idx + size].reshape(shape)
        weights.append(W)
        idx += size
    return weights

In [ ]:
def compute_gradient_vector(weights, X, Y_target):
    grads = compute_gradients(weights, X, Y_target)
    return np.concatenate([g.flatten() for g in grads])

In [ ]:
def compute_full_hessian(weights, X, Y_target, eps=HESSIAN_EPS):
    """Full Hessian via central finite differences on the gradient."""
    shapes = [W.shape for W in weights]
    theta = weights_to_vector(weights)
    n = len(theta)

    H = np.zeros((n, n))
    for i in range(n):
        theta_p = theta.copy()
        theta_m = theta.copy()
        theta_p[i] += eps
        theta_m[i] -= eps

        g_p = compute_gradient_vector(vector_to_weights(theta_p, shapes), X, Y_target)
        g_m = compute_gradient_vector(vector_to_weights(theta_m, shapes), X, Y_target)

        H[:, i] = (g_p - g_m) / (2 * eps)

    H = 0.5 * (H + H.T)
    return H

In [ ]:
def analyze_hessian(H):
    """Compute key Hessian statistics."""
    eigenvalues = np.linalg.eigvalsh(H)
    eigenvalues_sorted = np.sort(eigenvalues)[::-1]

    lambda_max = eigenvalues_sorted[0]
    lambda_min_pos = None
    trace_H = np.sum(eigenvalues)

    # lambda_min among positive eigenvalues
    pos_eigs = eigenvalues[eigenvalues > 1e-12]
    if len(pos_eigs) > 0:
        lambda_min_pos = np.min(pos_eigs)

    # Condition number
    if lambda_min_pos is not None and lambda_min_pos > 1e-15:
        kappa = lambda_max / lambda_min_pos
    else:
        kappa = np.inf

    return {
        'eigenvalues': eigenvalues_sorted,
        'lambda_max': lambda_max,
        'lambda_min_pos': lambda_min_pos,
        'trace': trace_H,
        'kappa': kappa,
    }

# Quick sanity check: analyze a known Hessian
_test_H = np.diag([10.0, 5.0, 2.0, 0.5])
_test_stats = analyze_hessian(_test_H)
print("Sanity check on diag([10, 5, 2, 0.5]):")
print(f"  lambda_max = {_test_stats['lambda_max']:.1f} (expected 10.0)")
print(f"  lambda_min_pos = {_test_stats['lambda_min_pos']:.1f} (expected 0.5)")
print(f"  kappa = {_test_stats['kappa']:.1f} (expected 20.0)")
print(f"  trace = {_test_stats['trace']:.1f} (expected 17.5)")
print(f"  num eigenvalues = {len(_test_stats['eigenvalues'])}")

## Single-Step Comparison: SGD vs Muon

### The Causal Isolation Protocol

This is the key methodological innovation of this experiment. Rather than training two networks separately (which would diverge in trajectory), we:

1. Train a **single** network with SGD to reach a checkpoint
2. From that **exact same checkpoint** (same weights, same momentum buffer), apply:
   - One SGD step --> measure Hessian change
   - One Muon step --> measure Hessian change
3. Compare the two Hessian changes

This **paired comparison from identical starting points** eliminates all confounds: same loss surface position, same gradient information, same momentum state. The only difference is the update rule itself.

**Key distinction in the update rules**:
- **SGD+momentum**: `W_new = W - lr * (momentum * velocity + grad)` -- the raw gradient direction, scaled by learning rate
- **Muon+momentum**: `W_new = W - lr * (momentum * velocity + NS(grad))` -- the gradient is first orthogonalized via Newton-Schulz, equalizing its singular values before the step

In [ ]:
def one_step_sgd(weights, X, Y_target, lr, momentum_state=None):
    """Take one SGD+momentum step. Returns new weights."""
    grads = compute_gradients(weights, X, Y_target)
    new_weights = []
    for i in range(len(weights)):
        if momentum_state is not None:
            vel = MOMENTUM * momentum_state[i] + grads[i]
        else:
            vel = grads[i]
        W_new = weights[i] - lr * vel
        new_weights.append(W_new)
    return new_weights

In [ ]:
def one_step_muon(weights, X, Y_target, lr, momentum_state=None):
    """Take one Muon step. Returns new weights."""
    grads = compute_gradients(weights, X, Y_target)
    new_weights = []
    for i in range(len(weights)):
        G_orth = newton_schulz_orthogonalize(grads[i], num_iters=NS_ITERS)
        if momentum_state is not None:
            vel = MOMENTUM * momentum_state[i] + G_orth
        else:
            vel = G_orth
        W_new = weights[i] - lr * vel
        new_weights.append(W_new)
    return new_weights

## Warmup Training (SGD to reach measurement points)

### Why SGD warmup, not Muon?

We deliberately use **SGD** for the warmup trajectory. This is important: if we warmed up with Muon, the snapshots would already reflect Muon's spectral regularization, and the single-step comparison would be confounded by the starting point being "Muon-friendly."

By warming up with SGD, we ensure the weight snapshots represent **generic** training points -- the kind of loss landscape geometry that SGD naturally visits. This makes the single-step comparison maximally informative: we are asking "starting from a landscape shaped by vanilla training, does one Muon step reshape curvature more than one SGD step?"

The warmup also provides snapshots at different **training phases**, from early (high loss, poorly conditioned) to late (lower loss, potentially better conditioned), letting us test whether Muon's per-step advantage depends on the optimization stage.

In [ ]:
def warmup_sgd(weights, X, Y_target, max_steps, measurement_steps):
    """
    Train with SGD, collecting weight snapshots and momentum states
    at specified measurement steps.
    """
    velocities = [np.zeros_like(W) for W in weights]
    snapshots = {}

    for step in range(max_steps):
        if step in measurement_steps:
            # Save deep copies of weights and momentum
            snapshots[step] = {
                'weights': [W.copy() for W in weights],
                'velocities': [v.copy() for v in velocities],
                'loss': compute_loss(weights, X, Y_target),
            }

        grads = compute_gradients(weights, X, Y_target)
        for i in range(len(weights)):
            velocities[i] = MOMENTUM * velocities[i] + grads[i]
            weights[i] = weights[i] - LR_SGD * velocities[i]

    return snapshots

## Main Experiment Loop

### Per-Seed Protocol

For each random seed, the experiment:

1. **Generates a random target function**: `Y = W2_target @ W1_target @ X`, so we have a ground-truth factorization the network must learn.
2. **Runs SGD warmup** for 1500 steps, collecting 10 weight/momentum snapshots.
3. **At each snapshot**, computes three 32x32 Hessians (before, after-SGD, after-Muon) and extracts the full eigenspectrum.
4. **Computes lifting ratios**: how much lambda_min changed from the single step.

The critical quantity is `lift_Muon / lift_SGD` -- the relative lambda_min lifting advantage of Muon over SGD in a single step. If this is consistently > 1 (and especially > 3), the per-step hypothesis is confirmed.

In [ ]:
def run_single_seed(seed):
    """Run the full per-step eigenvalue lifting test for one seed."""
    rng = np.random.RandomState(seed)

    # Generate data
    W_target = [rng.randn(DIM, DIM) * 0.3 for _ in range(NUM_LAYERS)]
    X = rng.randn(DIM, DATA_POINTS) * 0.5
    Y_target = X.copy()
    for W in W_target:
        Y_target = W @ Y_target

    print(f"    Data generated: X shape = {X.shape}, Y_target shape = {Y_target.shape}")
    print(f"    Target weight singular values: {[np.round(np.linalg.svd(W, compute_uv=False), 3).tolist() for W in W_target]}")

    # Initialize and warmup
    weights_init = init_weights(DIM, NUM_LAYERS, seed + 1000)
    print(f"    Initial weight singular values: {[np.round(np.linalg.svd(W, compute_uv=False), 3).tolist() for W in weights_init]}")

    initial_loss = compute_loss(weights_init, X, Y_target)
    print(f"    Initial loss: {initial_loss:.6f}")

    snapshots = warmup_sgd(
        [W.copy() for W in weights_init], X, Y_target,
        max_steps=WARMUP_MAX_STEPS,
        measurement_steps=MEASUREMENT_STEPS
    )

    print(f"    Warmup complete. Snapshots collected at steps: {sorted(snapshots.keys())}")
    print(f"    Loss trajectory during warmup: ", end="")
    for mstep in sorted(snapshots.keys()):
        print(f"step {mstep}: {snapshots[mstep]['loss']:.4e}  ", end="")
    print()

    results = []
    for mstep in MEASUREMENT_STEPS:
        if mstep not in snapshots:
            continue

        snap = snapshots[mstep]
        w_snap = snap['weights']
        v_snap = snap['velocities']
        loss_at = snap['loss']

        # Hessian BEFORE the step
        H_before = compute_full_hessian(w_snap, X, Y_target)
        stats_before = analyze_hessian(H_before)

        # One SGD step from this point
        w_after_sgd = one_step_sgd(w_snap, X, Y_target, LR_SGD, v_snap)
        H_after_sgd = compute_full_hessian(w_after_sgd, X, Y_target)
        stats_after_sgd = analyze_hessian(H_after_sgd)

        # One Muon step from the SAME point (same momentum state)
        w_after_muon = one_step_muon(w_snap, X, Y_target, LR_MUON, v_snap)
        H_after_muon = compute_full_hessian(w_after_muon, X, Y_target)
        stats_after_muon = analyze_hessian(H_after_muon)

        # Compute lifting ratios
        lmin_before = stats_before['lambda_min_pos']
        lmin_sgd = stats_after_sgd['lambda_min_pos']
        lmin_muon = stats_after_muon['lambda_min_pos']

        if lmin_before is not None and lmin_before > 1e-15:
            lift_sgd = lmin_sgd / lmin_before if lmin_sgd is not None else 0.0
            lift_muon = lmin_muon / lmin_before if lmin_muon is not None else 0.0
        else:
            lift_sgd = float('nan')
            lift_muon = float('nan')

        # Kappa change
        kappa_before = stats_before['kappa']
        kappa_sgd = stats_after_sgd['kappa']
        kappa_muon = stats_after_muon['kappa']

        if kappa_before > 0 and not np.isinf(kappa_before):
            kappa_ratio_sgd = kappa_sgd / kappa_before
            kappa_ratio_muon = kappa_muon / kappa_before
        else:
            kappa_ratio_sgd = float('nan')
            kappa_ratio_muon = float('nan')

        # Trace change
        tr_before = stats_before['trace']
        tr_sgd = stats_after_sgd['trace']
        tr_muon = stats_after_muon['trace']

        if abs(tr_before) > 1e-15:
            tr_ratio_sgd = tr_sgd / tr_before
            tr_ratio_muon = tr_muon / tr_before
        else:
            tr_ratio_sgd = float('nan')
            tr_ratio_muon = float('nan')

        # Print per-snapshot diagnostics
        print(f"    Step {mstep:>5}: loss={loss_at:.4e} | "
              f"lmin: {lmin_before:.3e} -> SGD:{lmin_sgd:.3e} Muon:{lmin_muon:.3e} | "
              f"lift SGD={lift_sgd:.4f} Muon={lift_muon:.4f} | "
              f"kappa: {kappa_before:.1f} -> SGD:{kappa_sgd:.1f} Muon:{kappa_muon:.1f}")

        results.append({
            'step': mstep,
            'loss': loss_at,
            'lmin_before': lmin_before,
            'lmin_sgd': lmin_sgd,
            'lmin_muon': lmin_muon,
            'lift_sgd': lift_sgd,
            'lift_muon': lift_muon,
            'kappa_before': kappa_before,
            'kappa_ratio_sgd': kappa_ratio_sgd,
            'kappa_ratio_muon': kappa_ratio_muon,
            'tr_ratio_sgd': tr_ratio_sgd,
            'tr_ratio_muon': tr_ratio_muon,
            'lmax_before': stats_before['lambda_max'],
            'lmax_sgd': stats_after_sgd['lambda_max'],
            'lmax_muon': stats_after_muon['lambda_max'],
        })

    return results

In [ ]:
def main():
    print()
    print("=" * 110)
    print("  Experiment 3.16: Eigenvalue Lifting -- Per-Step vs Cumulative")
    print("=" * 110)
    print()
    print("  QUESTION: Does a single Muon step lift lambda_min more than a single SGD step?")
    print()
    print(f"  Config: {NUM_LAYERS}-layer {DIM}x{DIM} deep linear net ({N_PARAMS} params)")
    print(f"  LR_SGD={LR_SGD}, LR_MUON={LR_MUON}, momentum={MOMENTUM}, NS_iters={NS_ITERS}")
    print(f"  Measurement points: {MEASUREMENT_STEPS}")
    print(f"  Averaging over {NUM_SEEDS} seeds")
    print()

    # =========================================================================
    # Run all seeds
    # =========================================================================
    all_results = []  # list of lists (per seed)

    for s in range(NUM_SEEDS):
        seed = 42 + s * 137
        print(f"  Running seed {s+1}/{NUM_SEEDS} (seed={seed})...", flush=True)
        seed_results = run_single_seed(seed)
        all_results.append(seed_results)

    # =========================================================================
    # Per-measurement-point aggregation
    # =========================================================================
    print()
    print("=" * 110)
    print("  PER-STEP EIGENVALUE LIFTING RESULTS (averaged over seeds)")
    print("=" * 110)
    print()

    print(f"  {'Step':>6} {'Loss':>10} {'lmin_bef':>10} "
          f"{'lift_SGD':>10} {'lift_Muon':>10} {'Muon/SGD':>10} "
          f"{'kR_SGD':>10} {'kR_Muon':>10} "
          f"{'trR_SGD':>10} {'trR_Muon':>10}")
    print(f"  {'-'*6} {'-'*10} {'-'*10} "
          f"{'-'*10} {'-'*10} {'-'*10} "
          f"{'-'*10} {'-'*10} "
          f"{'-'*10} {'-'*10}")

    # Organize results by measurement step
    step_to_data = {}
    for mstep in MEASUREMENT_STEPS:
        step_to_data[mstep] = {
            'losses': [], 'lmin_before': [],
            'lift_sgd': [], 'lift_muon': [],
            'kappa_ratio_sgd': [], 'kappa_ratio_muon': [],
            'tr_ratio_sgd': [], 'tr_ratio_muon': [],
        }

    for seed_results in all_results:
        for r in seed_results:
            mstep = r['step']
            if mstep in step_to_data:
                d = step_to_data[mstep]
                d['losses'].append(r['loss'])
                d['lmin_before'].append(r['lmin_before'] if r['lmin_before'] is not None else 0.0)
                if not np.isnan(r['lift_sgd']):
                    d['lift_sgd'].append(r['lift_sgd'])
                if not np.isnan(r['lift_muon']):
                    d['lift_muon'].append(r['lift_muon'])
                if not np.isnan(r['kappa_ratio_sgd']):
                    d['kappa_ratio_sgd'].append(r['kappa_ratio_sgd'])
                if not np.isnan(r['kappa_ratio_muon']):
                    d['kappa_ratio_muon'].append(r['kappa_ratio_muon'])
                if not np.isnan(r['tr_ratio_sgd']):
                    d['tr_ratio_sgd'].append(r['tr_ratio_sgd'])
                if not np.isnan(r['tr_ratio_muon']):
                    d['tr_ratio_muon'].append(r['tr_ratio_muon'])

    muon_vs_sgd_ratios = []  # lift_muon / lift_sgd per step

    for mstep in MEASUREMENT_STEPS:
        d = step_to_data[mstep]
        if len(d['lift_sgd']) == 0 or len(d['lift_muon']) == 0:
            continue

        loss_mean = np.mean(d['losses'])
        lmin_mean = np.mean(d['lmin_before'])
        lift_sgd_mean = np.mean(d['lift_sgd'])
        lift_muon_mean = np.mean(d['lift_muon'])

        if lift_sgd_mean > 1e-15:
            ratio = lift_muon_mean / lift_sgd_mean
        else:
            ratio = float('nan')
        muon_vs_sgd_ratios.append(ratio)

        kr_sgd = np.mean(d['kappa_ratio_sgd']) if d['kappa_ratio_sgd'] else float('nan')
        kr_muon = np.mean(d['kappa_ratio_muon']) if d['kappa_ratio_muon'] else float('nan')
        tr_sgd = np.mean(d['tr_ratio_sgd']) if d['tr_ratio_sgd'] else float('nan')
        tr_muon = np.mean(d['tr_ratio_muon']) if d['tr_ratio_muon'] else float('nan')

        print(f"  {mstep:>6} {loss_mean:10.4e} {lmin_mean:10.4e} "
              f"{lift_sgd_mean:10.4f} {lift_muon_mean:10.4f} {ratio:10.4f} "
              f"{kr_sgd:10.4f} {kr_muon:10.4f} "
              f"{tr_sgd:10.4f} {tr_muon:10.4f}")

    print()
    print("  Legend:")
    print("    lift_SGD/Muon = lambda_min(H_after) / lambda_min(H_before)  [>1 = lifting]")
    print("    Muon/SGD      = lift_Muon / lift_SGD  [>1 = Muon lifts more]")
    print("    kR             = kappa(H_after) / kappa(H_before)  [<1 = conditioning improves]")
    print("    trR            = tr(H_after) / tr(H_before)")

    # =========================================================================
    # Detailed per-seed table at a representative step
    # =========================================================================
    rep_step = 100  # representative step

    print()
    print("=" * 110)
    print(f"  DETAILED PER-SEED RESULTS AT STEP {rep_step}")
    print("=" * 110)
    print()

    print(f"  {'Seed':>6} {'Loss':>10} {'lmin_bef':>10} "
          f"{'lmin_SGD':>10} {'lmin_Muon':>10} "
          f"{'lift_SGD':>10} {'lift_Muon':>10} {'Muon>SGD?':>10}")
    print(f"  {'-'*6} {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")

    for s, seed_results in enumerate(all_results):
        for r in seed_results:
            if r['step'] == rep_step:
                muon_wins = "YES" if r['lift_muon'] > r['lift_sgd'] else "NO"
                print(f"  {s+1:>6} {r['loss']:10.4e} {r['lmin_before']:10.4e} "
                      f"{r['lmin_sgd']:10.4e} {r['lmin_muon']:10.4e} "
                      f"{r['lift_sgd']:10.4f} {r['lift_muon']:10.4f} {muon_wins:>10}")

    # =========================================================================
    # Lambda_max analysis (does Muon also reduce sharpness per step?)
    # =========================================================================
    print()
    print("=" * 110)
    print("  LAMBDA_MAX (SHARPNESS) PER-STEP CHANGE")
    print("=" * 110)
    print()

    print(f"  {'Step':>6} {'lmax_bef':>12} {'lmax_SGD':>12} {'lmax_Muon':>12} "
          f"{'SGD ratio':>12} {'Muon ratio':>12}")
    print(f"  {'-'*6} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")

    for mstep in MEASUREMENT_STEPS:
        lmax_bef_list = []
        lmax_sgd_list = []
        lmax_muon_list = []
        for seed_results in all_results:
            for r in seed_results:
                if r['step'] == mstep:
                    lmax_bef_list.append(r['lmax_before'])
                    lmax_sgd_list.append(r['lmax_sgd'])
                    lmax_muon_list.append(r['lmax_muon'])

        if not lmax_bef_list:
            continue

        lmax_bef_mean = np.mean(lmax_bef_list)
        lmax_sgd_mean = np.mean(lmax_sgd_list)
        lmax_muon_mean = np.mean(lmax_muon_list)

        r_sgd = lmax_sgd_mean / lmax_bef_mean if lmax_bef_mean > 1e-15 else float('nan')
        r_muon = lmax_muon_mean / lmax_bef_mean if lmax_bef_mean > 1e-15 else float('nan')

        print(f"  {mstep:>6} {lmax_bef_mean:12.4e} {lmax_sgd_mean:12.4e} {lmax_muon_mean:12.4e} "
              f"{r_sgd:12.4f} {r_muon:12.4f}")

    # =========================================================================
    # HYPOTHESIS TESTS
    # =========================================================================
    print()
    print("=" * 110)
    print("  HYPOTHESIS TESTS")
    print("=" * 110)
    print()

    # Test 1: Muon lifts lambda_min more than SGD at majority of measurement points
    valid_ratios = [r for r in muon_vs_sgd_ratios if not np.isnan(r)]
    muon_wins_count = sum(1 for r in valid_ratios if r > 1.0)
    total_points = len(valid_ratios)

    t1_pass = muon_wins_count > total_points * 0.5
    print(f"  T1: Muon lifts lambda_min more than SGD at majority of points?")
    print(f"      Points where Muon/SGD > 1: {muon_wins_count}/{total_points}")
    print(f"      {'PASS' if t1_pass else 'FAIL'}")
    print()

    # Test 2: KEY TEST -- Muon lifts lambda_min by >3x what SGD does (on average)
    if valid_ratios:
        mean_ratio = np.mean(valid_ratios)
        median_ratio = np.median(valid_ratios)
    else:
        mean_ratio = float('nan')
        median_ratio = float('nan')

    t2_pass = mean_ratio > 3.0
    print(f"  T2: KEY TEST -- Muon lifts lambda_min by >3x vs SGD on average?")
    print(f"      Mean lift_Muon/lift_SGD = {mean_ratio:.4f}")
    print(f"      Median = {median_ratio:.4f}")
    print(f"      {'PASS' if t2_pass else 'FAIL'}: {'>' if t2_pass else '<='} 3x threshold")
    print()

    # Test 3: Muon reduces kappa per step more than SGD
    kappa_muon_better = 0
    kappa_total = 0
    for mstep in MEASUREMENT_STEPS:
        d = step_to_data[mstep]
        if d['kappa_ratio_sgd'] and d['kappa_ratio_muon']:
            kr_sgd = np.mean(d['kappa_ratio_sgd'])
            kr_muon = np.mean(d['kappa_ratio_muon'])
            if kr_muon < kr_sgd:
                kappa_muon_better += 1
            kappa_total += 1

    t3_pass = kappa_muon_better > kappa_total * 0.5
    print(f"  T3: Muon reduces kappa per step more than SGD at majority of points?")
    print(f"      Points where Muon kappa ratio < SGD kappa ratio: {kappa_muon_better}/{kappa_total}")
    print(f"      {'PASS' if t3_pass else 'FAIL'}")
    print()

    # Test 4: Effect is consistent across seeds (low variance)
    if valid_ratios:
        cv = np.std(valid_ratios) / np.mean(valid_ratios) if np.mean(valid_ratios) > 0 else float('inf')
        t4_pass = cv < 1.0  # coefficient of variation < 1 means reasonably consistent
    else:
        cv = float('nan')
        t4_pass = False
    print(f"  T4: Effect is consistent (CV of Muon/SGD ratio < 1.0)?")
    print(f"      Coefficient of variation = {cv:.4f}")
    print(f"      {'PASS' if t4_pass else 'FAIL'}")
    print()

    # Test 5: The per-step effect persists late in training (not just early)
    late_ratios = []
    for mstep in MEASUREMENT_STEPS:
        if mstep >= 600:
            d = step_to_data[mstep]
            if d['lift_sgd'] and d['lift_muon']:
                ls = np.mean(d['lift_sgd'])
                lm = np.mean(d['lift_muon'])
                if ls > 1e-15:
                    late_ratios.append(lm / ls)

    if late_ratios:
        mean_late = np.mean(late_ratios)
        t5_pass = mean_late > 1.0
    else:
        mean_late = float('nan')
        t5_pass = False
    print(f"  T5: Per-step lifting persists late in training (steps >= 600)?")
    print(f"      Mean Muon/SGD ratio for late steps: {mean_late:.4f}")
    print(f"      {'PASS' if t5_pass else 'FAIL'}")
    print()

    # =========================================================================
    # OVERALL VERDICT
    # =========================================================================
    print("=" * 110)
    print("  OVERALL VERDICT")
    print("=" * 110)
    print()

    if t2_pass:
        print("  [CONFIRMED] Muon lifts lambda_min by >3x per step compared to SGD.")
        print(f"  Mean lifting ratio = {mean_ratio:.2f}x.")
        print("  The 330x kappa reduction at convergence is (at least partly) a PER-STEP effect.")
    elif t1_pass:
        print(f"  [PARTIAL] Muon lifts lambda_min more than SGD ({mean_ratio:.2f}x) but not >3x.")
        print("  The kappa reduction may be partly per-step, partly cumulative.")
    else:
        print(f"  [REJECTED] Muon does NOT lift lambda_min more than SGD per step ({mean_ratio:.2f}x).")
        print("  The 330x kappa reduction is a CUMULATIVE effect, not per-step.")

    if t3_pass:
        print("  [BONUS] Muon also reduces condition number more per step.")

    if t5_pass:
        print("  [BONUS] The per-step effect persists late in training.")
    elif t1_pass:
        print("  [NOTE] The per-step effect may weaken late in training.")

    print()
    print("=" * 110)
    print("  EXPERIMENT COMPLETE")
    print("=" * 110)

## Results Aggregation and Hypothesis Testing

The `main()` function below orchestrates the full experiment: running all seeds, aggregating results across seeds at each measurement point, and then evaluating five hypothesis tests:

| Test | Question | Threshold |
|------|----------|-----------|
| **T1** | Does Muon lift lambda_min more than SGD at a majority of measurement points? | > 50% of points |
| **T2** (KEY) | Does Muon lift lambda_min by > 3x what SGD does on average? | mean ratio > 3.0 |
| **T3** | Does Muon reduce kappa per step more than SGD? | majority of points |
| **T4** | Is the effect consistent across seeds (low variance)? | CV < 1.0 |
| **T5** | Does the per-step effect persist late in training (steps >= 600)? | mean ratio > 1.0 |

### Interpreting the Results Table

- **lift_SGD / lift_Muon**: ratio of lambda_min after one step to lambda_min before. Values > 1 mean the optimizer lifted the smallest eigenvalue (good for conditioning). Values < 1 mean it suppressed it.
- **Muon/SGD**: the ratio of lift_Muon to lift_SGD. This is the primary test statistic. Values > 1 favor Muon; values > 3 strongly confirm the per-step hypothesis.
- **kR (kappa ratio)**: how much the condition number changed in one step. Values < 1 mean improved conditioning.
- **trR (trace ratio)**: how much the total Hessian curvature changed. Tracks whether the optimizer increases or decreases overall sharpness.

In [ ]:
if __name__ == '__main__':
    main()

## Conclusions and Scientific Interpretation

### What This Experiment Reveals About Muon's Mechanism

This experiment answers a fundamental mechanistic question: **Is Muon's dramatic improvement in Hessian conditioning (330x kappa reduction at convergence) driven by per-step spectral reshaping, or is it purely a cumulative trajectory effect?**

#### Possible Outcomes and Their Implications

1. **If T2 PASSES (Muon/SGD > 3x)**: Each individual Muon step acts as a **spectral equalizer** on the loss landscape. The Newton-Schulz orthogonalization doesn't just change the step direction -- it actively lifts the smallest Hessian eigenvalues, compressing the eigenvalue spectrum from below. This would mean Muon functions as a **per-step curvature regularizer**, and the 330x cumulative effect is an amplification of this per-step mechanism.

2. **If T1 PASSES but T2 FAILS (Muon beats SGD but < 3x)**: Muon has a modest per-step advantage that compounds over training. The 330x factor arises from many small per-step improvements accumulating multiplicatively. The mechanism is real but subtle -- more like compound interest than a single dramatic effect.

3. **If T1 FAILS (SGD lifts lambda_min as well or better per step)**: The 330x kappa reduction is entirely a **trajectory effect** -- Muon's orthogonal updates steer the optimization toward regions of parameter space that happen to have better-conditioned Hessians, but any single step does not preferentially improve conditioning. This would suggest the mechanism is about **path selection** rather than **per-step curvature control**.

### Connection to RG Gauge-Fixing Interpretation

In the renormalization group (RG) gauge-fixing interpretation of Muon, the Newton-Schulz orthogonalization removes redundant degrees of freedom (gauge modes) at each step. If eigenvalue lifting is per-step, it supports the interpretation that gauge-fixing directly regularizes the curvature structure -- removing gauge modes lifts the flat directions (small eigenvalues) of the Hessian because those flat directions correspond to gauge orbits.

### Key Caveats

- **Deep linear networks only**: The Hessian structure of deep linear nets differs from nonlinear networks. The absence of activation functions means all curvature comes from the product structure `W_2 W_1`, not from nonlinear interactions.
- **Small scale**: 32 parameters is far from realistic networks. The effect may change at scale.
- **Fixed momentum state**: We use the SGD momentum buffer for both optimizers. In practice, Muon would accumulate its own momentum, which could further amplify the per-step effect.
- **LR ratio**: The 2x LR ratio (Muon 0.02 vs SGD 0.01) means Muon takes larger steps in parameter space. Part of any lambda_min lifting could be a step-size effect rather than a direction effect. Experiment H6a (LR Confound Audit) addresses this directly.